# CMP Removal Rate Prediction

This project uses the PHM Data Challenge 2016 CMP dataset to predict average material removal rate (`AVG_REMOVAL_RATE`).

The workflow includes data cleaning, wafer-stage feature engineering, exploratory analysis, feature selection,
group-specific model training, and model evaluation.

Reusable functions are implemented in the accompanying Python modules. This notebook defines the experiment settings, executes
the workflow, and presents the results.

## 1. Environment and Imports

Run this notebook from the project directory containing the Python modules.

Select a Python environment with all required dependencies installed, then execute the cells sequentially from top to bottom.

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from IPython.display import display
from pandas.errors import PerformanceWarning
from sklearn.preprocessing import RobustScaler

from data import (
    list_csv_files_in_local_folder,
    load_single_csv_from_local,
    standardize_column_names,
    standardize_stage_value,
    standardize_wafer_id,
    standardize_keys,
    prepare_label_df_strict,
    merge_features_with_labels_by_keys,
)

from data_cleaning import (
    clean_series_by_local_outlier_then_fill,
    consolidate_duplicate_compressed_keys,
    get_drop_keys_by_target_range,
    drop_rows_by_keys,
    compute_iqr_bounds,
    detect_target_outliers_iqr,
    remove_target_outliers_iqr,
    flag_target_outliers_by_bounds,
    fit_preprocessor_from_train_best,
    apply_preprocessor_best,
)

In [ ]:
from feature_engineering import (
    safe_numeric_series,
    first_valid_value,
    last_valid_value,
    safe_quantile,
    slope_feature,
    lag1_autocorr_feature,
    first_half_mean,
    second_half_mean,
    segment_slices,
    segment_mean,
    segment_std,
    segment_median,
    upper_tail_ratio,
    spike_count,
    low_flow_ratio,
    stability_feature,
    detect_feature_profile,
    get_stage_stat_plan,
    compress_one_group_best,
    compress_timeseries_folder_incremental_best,
    feature_engineering_compressed_best,
)

from feature_selection import (
    feature_complexity_score,
    prefilter_features_by_correlation_global,
    build_feature_correlation_table,
    select_features_by_cumulative_importance,
)

from model_building import (
    transform_target,
    inverse_target_transform,
    suggest_params,
    fit_xgb_native,
    predict_xgb_native,
    build_sklearn_pipeline,
    fit_model,
    predict_model,
    normalize_weight_dict,
    apply_weighted_ensemble,
    rmse,
    mae,
    mse,
    squared_error_stats,
)

In [ ]:
from training import (
    sliding_window_split,
    tune_model_with_bayes,
    train_groupwise_models,
    predict_by_true_group,
    build_prediction_map,
    tune_groupwise_ensemble_weights,
)

from visualization import (
    safe_corr,
    pick_cols_by_keywords,
    prepare_eda_tables,
    plot_target_fixed_version,
    plot_validation_target_preserved,
    plot_stage_distribution,
    plot_eda_correlation_heatmap,
    plot_eda_feature_scatter,
    plot_scatter,
    get_xgb_feature_importance,
    get_rf_feature_importance,
    get_svr_permutation_importance,
    plot_group_feature_importance,
)

warnings.simplefilter("ignore", PerformanceWarning)

optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.simplefilter("ignore", PerformanceWarning)

pd.set_option("display.max_columns", 50)

pd.set_option("display.width", 200)

pd.set_option("display.max_colwidth", 120)

## 2. Data Paths and Preprocessing Settings

The input data is expected under `CMP-data/` in the project root. The test process files are stored in `CMP-data/testing/`, matching the GitHub repository structure.

The directory must contain `training/`, `validation/`, and `testing/` subdirectories, together with their corresponding removal-rate label files.
Generated CSV tables are saved under `results/tables/`.

Raw-data crosstabs reread all training files even if MAX_FILES is changed for compression.

In [ ]:
PROJECT_ROOT = Path.cwd()

BASE_DIR = PROJECT_ROOT / "CMP-data"

TABLES_DIR = PROJECT_ROOT / "results" / "tables"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FOLDER = BASE_DIR / "training"
VALID_FOLDER = BASE_DIR / "validation"
TEST_FOLDER = BASE_DIR / "testing"

TRAIN_LABEL_FILE = BASE_DIR / "CMP-training-removalrate.csv"
VALID_LABEL_FILE = BASE_DIR / "CMP-validation-removalrate.csv"
TEST_LABEL_FILE = BASE_DIR / "CMP-test-removalrate.csv"

MAX_FILES = None

TARGET_COL = "AVG_REMOVAL_RATE"

KEY_COLS = ["WAFER_ID", "STAGE"]

LABEL_CONSISTENCY_TOL = 1e-8

DROP_INCONSISTENT_LABEL_KEYS = True

SEGMENT_COUNT = 3

IQR_CLIP_MULTIPLIER = 4.0

TARGET_OUTLIER_K = 10

SHOW_FILE_PROGRESS = False

SHOW_SUMMARY_PROGRESS = True

def print_target_describe(df: pd.DataFrame, target_col=TARGET_COL, name="Dataset"):
    print(f"\n=== {name} target describe ===")
    print(pd.to_numeric(df[target_col], errors="coerce").describe())

## 3. Time-Series Cleaning and Feature Engineering

Each CSV file is processed separately. Observations are grouped by `WAFER_ID` and `STAGE` and ordered by timestamp when available.

Within each group, local outliers are masked using a 3 × IQR rule, followed by linear interpolation, forward filling, and backward filling.

The time series are summarized using distribution statistics, trends, and process-specific features. Selected signal types also receive statistics from three positional segments.

Repeated wafer-stage keys across files are consolidated after feature extraction.

The local IQR rule requires at least five valid values and a positive IQR. Segments are positional, not equal-duration intervals. The current feature plan is identical for stages A and B. Numeric compressed features are averaged across duplicate keys, SOURCE_FILE_NUNIQUE is summed, and non-numeric values use the first row; this differs from recomputing features on concatenated raw signals.

In [ ]:
required_paths = [TRAIN_FOLDER, VALID_FOLDER, TEST_FOLDER,
                  TRAIN_LABEL_FILE, VALID_LABEL_FILE, TEST_LABEL_FILE]

missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Check BASE_DIR and the original CMP data layout. Missing: "
        + ", ".join(missing_paths)
    )

print("=== Path Check ===")

print("TRAIN_FOLDER exists:", TRAIN_FOLDER.exists(), TRAIN_FOLDER)

print("VALID_FOLDER exists:", VALID_FOLDER.exists(), VALID_FOLDER)

print("TRAIN_LABEL_FILE exists:", TRAIN_LABEL_FILE.exists(), TRAIN_LABEL_FILE)

print("VALID_LABEL_FILE exists:", VALID_LABEL_FILE.exists(), VALID_LABEL_FILE)

train_compressed = compress_timeseries_folder_incremental_best(
    TRAIN_FOLDER,
    dataset_name="Training",
    verbose=False,
    max_files=MAX_FILES,
    target_col=TARGET_COL,
    segment_count=SEGMENT_COUNT,
    show_file_progress=SHOW_FILE_PROGRESS,
    show_summary_progress=SHOW_SUMMARY_PROGRESS,
)

valid_compressed = compress_timeseries_folder_incremental_best(
    VALID_FOLDER,
    dataset_name="Validation",
    verbose=False,
    max_files=MAX_FILES,
    target_col=TARGET_COL,
    segment_count=SEGMENT_COUNT,
    show_file_progress=SHOW_FILE_PROGRESS,
    show_summary_progress=SHOW_SUMMARY_PROGRESS,
)

In [ ]:
test_compressed = compress_timeseries_folder_incremental_best(
    TEST_FOLDER,
    dataset_name="Test",
    verbose=False,
    max_files=MAX_FILES,
    target_col=TARGET_COL,
    segment_count=SEGMENT_COUNT,
    show_file_progress=SHOW_FILE_PROGRESS,
    show_summary_progress=SHOW_SUMMARY_PROGRESS,
)

train_compressed = consolidate_duplicate_compressed_keys(train_compressed, dataset_name="Training")

valid_compressed = consolidate_duplicate_compressed_keys(valid_compressed, dataset_name="Validation")

test_compressed = consolidate_duplicate_compressed_keys(test_compressed, dataset_name="Test")

print("\nCompressed shape:")

print("Train:", train_compressed.shape)

print("Valid:", valid_compressed.shape)

## 4. Label Preparation and Alignment

Column names and wafer-stage keys are standardized before matching features with labels.

For repeated label keys, consistency is assessed using the difference between the maximum and minimum target values.
Inconsistent groups are removed under the configured tolerance, and retained groups are represented by their median target.

Samples outside the inclusive target range [0, 200] are removed from training, validation, and test data.

Features and labels are merged using a one-to-one left join on `WAFER_ID` and `STAGE`. Rows without valid target values are removed afterward.

In [ ]:
train_label_df = load_single_csv_from_local(TRAIN_LABEL_FILE)

valid_label_df = load_single_csv_from_local(VALID_LABEL_FILE)

test_label_df = load_single_csv_from_local(TEST_LABEL_FILE)

train_label_clean = prepare_label_df_strict(
    train_label_df,
    target_col=TARGET_COL,
    tol=LABEL_CONSISTENCY_TOL,
    drop_inconsistent=DROP_INCONSISTENT_LABEL_KEYS,
)

valid_label_clean = prepare_label_df_strict(
    valid_label_df,
    target_col=TARGET_COL,
    tol=LABEL_CONSISTENCY_TOL,
    drop_inconsistent=DROP_INCONSISTENT_LABEL_KEYS,
)

test_label_clean = prepare_label_df_strict(
    test_label_df,
    target_col=TARGET_COL,
    tol=LABEL_CONSISTENCY_TOL,
    drop_inconsistent=DROP_INCONSISTENT_LABEL_KEYS,
)

print("\nLabel clean shape:")

print("Train:", train_label_clean.shape)

print("Valid:", valid_label_clean.shape)

train_drop_rows, train_drop_keys = get_drop_keys_by_target_range(train_label_clean, target_col=TARGET_COL, lower=0, upper=200)

valid_drop_rows, valid_drop_keys = get_drop_keys_by_target_range(valid_label_clean, target_col=TARGET_COL, lower=0, upper=200)

test_drop_rows, test_drop_keys = get_drop_keys_by_target_range(test_label_clean, target_col=TARGET_COL, lower=0, upper=200)

print("\n=== Drop MRR Outside [0, 200] Summary ===")

print("Train drop key count:", len(train_drop_keys))

print("Valid drop key count:", len(valid_drop_keys))

print("Test drop key count :", len(test_drop_keys))

display(train_drop_rows.sort_values(TARGET_COL, ascending=False))

display(valid_drop_rows.sort_values(TARGET_COL, ascending=False))

In [ ]:
display(test_drop_rows.sort_values(TARGET_COL, ascending=False))

train_compressed = drop_rows_by_keys(train_compressed, train_drop_keys)

valid_compressed = drop_rows_by_keys(valid_compressed, valid_drop_keys)

test_compressed = drop_rows_by_keys(test_compressed, test_drop_keys)

train_label_clean = drop_rows_by_keys(train_label_clean, train_drop_keys)

valid_label_clean = drop_rows_by_keys(valid_label_clean, valid_drop_keys)

test_label_clean = drop_rows_by_keys(test_label_clean, test_drop_keys)

train_merged = merge_features_with_labels_by_keys(
    train_compressed,
    train_label_clean,
    target_col=TARGET_COL,
    dataset_name="Training"
)

valid_merged = merge_features_with_labels_by_keys(
    valid_compressed,
    valid_label_clean,
    target_col=TARGET_COL,
    dataset_name="Validation"
)

test_merged = merge_features_with_labels_by_keys(
    test_compressed,
    test_label_clean,
    target_col=TARGET_COL,
    dataset_name="Test"
)

train_merged = train_merged.dropna(subset=[TARGET_COL]).reset_index(drop=True)

valid_merged = valid_merged.dropna(subset=[TARGET_COL]).reset_index(drop=True)

test_merged = test_merged.dropna(subset=[TARGET_COL]).reset_index(drop=True)

### 4.1 Additional Training-Target Outlier Filtering

Training receives an additional target filter using the wider bounds from 10 x IQR and the 0.001/0.999 quantiles. Training outliers are removed. Validation rows beyond these training bounds are flagged but retained; test rows receive no additional removal by these bounds.

Validation is preserved relative to this step only, after earlier label and range filtering. The plotting helper prints plain-IQR diagnostics, which can differ from the actual removal bounds, and uses 45 histogram edges regardless of its bins argument.

In [ ]:
print_target_describe(train_merged, TARGET_COL, "Training merged BEFORE target outlier removal")

train_merged_clean, train_target_outliers, train_outlier_mask, lower_y, upper_y = remove_target_outliers_iqr(
    train_merged,
    target_col=TARGET_COL,
    k=TARGET_OUTLIER_K
)

print_target_describe(train_merged_clean, TARGET_COL, "Training merged AFTER target outlier removal")

target_filter_fig = plot_target_fixed_version(
    train_merged,
    train_merged_clean,
    target_col=TARGET_COL,
    bins=30,
    outlier_k=TARGET_OUTLIER_K
)

target_filter_fig.savefig(
    FIGURES_DIR / "training_target_outlier_filter.png",
    dpi=300,
    bbox_inches="tight",
)

print_target_describe(valid_merged, TARGET_COL, "Validation ORIGINAL (distribution preserved)")

valid_y_numeric = pd.to_numeric(valid_merged[TARGET_COL], errors="coerce")

valid_outlier_mask = ((valid_y_numeric < lower_y) | (valid_y_numeric > upper_y)).fillna(False)

valid_merged_final = valid_merged.copy()

valid_target_outliers = valid_merged_final.loc[
    valid_outlier_mask,
    ["WAFER_ID", "STAGE", TARGET_COL]
].copy()

validation_target_fig = plot_validation_target_preserved(
    valid_merged_final[TARGET_COL],
    target_col=TARGET_COL,
    bins=30
)

validation_target_fig.savefig(
    FIGURES_DIR / "validation_target_distribution.png",
    dpi=300,
    bbox_inches="tight",
)


## 5. Feature-Level Preprocessing

Feature outlier bounds are estimated from the training data using a 4 × IQR rule and applied to all three datasets.

Out-of-bound values are replaced with missing values rather than clipped to the bounds. Missing values are interpolated and filled
in the existing row order within each supplied dataset.

The feature matrices are aligned to the training columns.
Infinite values are converted to missing values, and columns that are entirely missing in training are removed from all splits.

RobustScaler is fitted on the training feature matrix and then applied to validation and test data.

This interpolation is across existing feature-table rows without grouping or chronological sorting. Numeric metadata, including START_TIME_ORD, undergoes this cleaning and scaling too. Its resulting values are later used for time ordering, so raw chronology is not guaranteed to be preserved.

In [ ]:
prep = fit_preprocessor_from_train_best(
    train_merged_clean,
    target_col=TARGET_COL,
    iqr_clip_multiplier=IQR_CLIP_MULTIPLIER
)

train_processed = apply_preprocessor_best(train_merged_clean, prep, target_col=TARGET_COL)

valid_processed = apply_preprocessor_best(valid_merged_final, prep, target_col=TARGET_COL)

test_processed = apply_preprocessor_best(test_merged, prep, target_col=TARGET_COL)

print("\nProcessed shape:")

print("Train:", train_processed.shape)

print("Valid:", valid_processed.shape)

train_fe = train_processed.copy()

valid_fe = valid_processed.copy()

test_fe = test_processed.copy()

train_keys = train_fe[["WAFER_ID", "STAGE"]].copy().reset_index(drop=True)

valid_keys = valid_fe[["WAFER_ID", "STAGE"]].copy().reset_index(drop=True)

test_keys = test_fe[["WAFER_ID", "STAGE"]].copy().reset_index(drop=True)

drop_non_feature_cols = ["STAGE", "WAFER_ID", TARGET_COL]

X_train = train_fe.drop(columns=drop_non_feature_cols, errors="ignore")

y_train = train_fe[TARGET_COL].copy()

X_valid = valid_fe.drop(columns=drop_non_feature_cols, errors="ignore")

y_valid = valid_fe[TARGET_COL].copy()

X_test = test_fe.drop(columns=drop_non_feature_cols, errors="ignore")

y_test = test_fe[TARGET_COL].copy()

X_train = X_train.select_dtypes(include=[np.number]).copy()

X_valid = X_valid.select_dtypes(include=[np.number]).copy()

X_test = X_test.select_dtypes(include=[np.number]).copy()

In [ ]:
for c in X_train.columns:
    if c not in X_valid.columns:
        X_valid[c] = np.nan
    if c not in X_test.columns:
        X_test[c] = np.nan

extra_valid_cols = [c for c in X_valid.columns if c not in X_train.columns]

if len(extra_valid_cols) > 0:
    X_valid = X_valid.drop(columns=extra_valid_cols)

extra_test_cols = [c for c in X_test.columns if c not in X_train.columns]

if len(extra_test_cols) > 0:
    X_test = X_test.drop(columns=extra_test_cols)

X_valid = X_valid[X_train.columns].copy()

X_test = X_test[X_train.columns].copy()

print("\n=== X / y Shape ===")

print("X_train:", X_train.shape)

print("y_train:", y_train.shape)

print("X_valid:", X_valid.shape)

print("y_valid:", y_valid.shape)

print("X_test:", X_test.shape)

print("y_test:", y_test.shape)

X_train_selected = X_train.copy().replace([np.inf, -np.inf], np.nan)

X_valid_selected = X_valid.copy().replace([np.inf, -np.inf], np.nan)

X_test_selected = X_test.copy().replace([np.inf, -np.inf], np.nan)

all_nan_cols = X_train_selected.columns[X_train_selected.isna().all()].tolist()

X_train_selected = X_train_selected.drop(columns=all_nan_cols)

X_valid_selected = X_valid_selected.drop(columns=[c for c in all_nan_cols if c in X_valid_selected.columns])

X_test_selected = X_test_selected.drop(columns=[c for c in all_nan_cols if c in X_test_selected.columns])

selected_cols = X_train_selected.columns.tolist()

In [ ]:
print("\n=== Selected Feature Matrix ===")

print("Train shape:", X_train_selected.shape)

print("Valid shape:", X_valid_selected.shape)

print("Test shape :", X_test_selected.shape)

print("Total selected features:", len(selected_cols))

selected_feature_df = pd.DataFrame({"selected_feature": selected_cols})

scaler = RobustScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_selected),
    columns=X_train_selected.columns,
    index=X_train_selected.index
)

X_valid_scaled = pd.DataFrame(
    scaler.transform(X_valid_selected),
    columns=X_valid_selected.columns,
    index=X_valid_selected.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_selected),
    columns=X_test_selected.columns,
    index=X_test_selected.index
)

print("\n=== Robust Scaled Feature Matrix ===")

print("Train shape:", X_train_scaled.shape)

print("Valid shape:", X_valid_scaled.shape)

print("Test shape :", X_test_scaled.shape)

final_train_dataset = X_train_scaled.copy()

final_train_dataset[TARGET_COL] = y_train.values

final_valid_dataset = X_valid_scaled.copy()

final_valid_dataset[TARGET_COL] = y_valid.values

final_test_dataset = X_test_scaled.copy()

In [ ]:
final_test_dataset[TARGET_COL] = y_test.values

print("\n=== Final Dataset Shape ===")

print("Final Train:", final_train_dataset.shape)

print("Final Valid:", final_valid_dataset.shape)

print("Final Test :", final_test_dataset.shape)

import pandas as pd

import numpy as np

from IPython.display import display

## 6. Data Summary and Quality Checks

The following summaries describe sample counts, feature counts, equipment and stage distributions, duplicate keys, missing values, and low-variation features.

Each result should be interpreted at the processing stage of the table being inspected. In particular, the absence of missing
values after preprocessing does not imply that the raw data contained no missing values.

Constant and near-zero-variance features are reported here; these checks do not automatically remove them.

Variable counts include the target. Equipment crosstabs count raw time-step rows, not wafer-stage samples. Target summaries use cleaned label tables and may include training labels excluded by the additional IQR filter. Duplicate checks use already-consolidated tables. The extreme-target report includes retained validation flags as well as removed samples.

In [ ]:
dataset_summary_df = pd.DataFrame([
    {"dataset": "train", "sample_count": final_train_dataset.shape[0], "variable_count": final_train_dataset.shape[1]},
    {"dataset": "valid", "sample_count": final_valid_dataset.shape[0], "variable_count": final_valid_dataset.shape[1]},
    {"dataset": "test",  "sample_count": final_test_dataset.shape[0],  "variable_count": final_test_dataset.shape[1]},
])

print("\n=== Sample Size and Variable Count ===")

print(dataset_summary_df.to_string(index=False))

try:
    train_files = list_csv_files_in_local_folder(TRAIN_FOLDER)

    raw_train_rows = []
    for fp in train_files:
        df_tmp = pd.read_csv(fp)
        df_tmp = standardize_column_names(df_tmp)
        df_tmp = standardize_keys(df_tmp)

        if "CHAMBER" in df_tmp.columns and "STAGE" in df_tmp.columns:
            raw_train_rows.append(df_tmp[["CHAMBER", "STAGE"]].copy())

    if len(raw_train_rows) > 0:
        raw_train_stage_df = pd.concat(raw_train_rows, axis=0).reset_index(drop=True)
        raw_train_stage_df["CHAMBER"] = pd.to_numeric(raw_train_stage_df["CHAMBER"], errors="coerce")
        raw_train_stage_df["STAGE"] = raw_train_stage_df["STAGE"].astype(str).str.strip().str.upper()

        chamber_stage_ct = pd.crosstab(raw_train_stage_df["CHAMBER"], raw_train_stage_df["STAGE"])

        print("\n=== CHAMBER x STAGE Crosstab ===")
        display(chamber_stage_ct)
    else:
        chamber_stage_ct = pd.DataFrame()
        print("\n=== CHAMBER x STAGE Crosstab ===")
        print("No CHAMBER/STAGE columns found in raw training csv files.")
except Exception as e:
    chamber_stage_ct = pd.DataFrame()
    print("\n=== CHAMBER x STAGE Crosstab ===")
    print(f"Unable to build crosstab: {e}")

In [ ]:
try:
    train_files = list_csv_files_in_local_folder(TRAIN_FOLDER)

    raw_train_rows = []
    for fp in train_files:
        df_tmp = pd.read_csv(fp)
        df_tmp = standardize_column_names(df_tmp)
        df_tmp = standardize_keys(df_tmp)

        if "MACHINE_DATA" in df_tmp.columns and "CHAMBER" in df_tmp.columns:
            raw_train_rows.append(df_tmp[["MACHINE_DATA", "CHAMBER"]].copy())

    if len(raw_train_rows) > 0:
        raw_train_machine_df = pd.concat(raw_train_rows, axis=0).reset_index(drop=True)
        raw_train_machine_df["MACHINE_DATA"] = pd.to_numeric(raw_train_machine_df["MACHINE_DATA"], errors="coerce")
        raw_train_machine_df["CHAMBER"] = pd.to_numeric(raw_train_machine_df["CHAMBER"], errors="coerce")

        machine_chamber_ct = pd.crosstab(raw_train_machine_df["MACHINE_DATA"], raw_train_machine_df["CHAMBER"])

        print("\n=== MACHINE_DATA x CHAMBER Crosstab ===")
        display(machine_chamber_ct)
    else:
        machine_chamber_ct = pd.DataFrame()
        print("\n=== MACHINE_DATA x CHAMBER Crosstab ===")
        print("No MACHINE_DATA/CHAMBER columns found in raw training csv files.")
except Exception as e:
    machine_chamber_ct = pd.DataFrame()
    print("\n=== MACHINE_DATA x CHAMBER Crosstab ===")
    print(f"Unable to build crosstab: {e}")

target_summary_rows = []

for dataset_name, label_df in {
    "train": train_label_clean,
    "valid": valid_label_clean,
    "test": test_label_clean,
}.items():
    y = pd.to_numeric(label_df[TARGET_COL], errors="coerce").dropna()

    target_summary_rows.append({
        "dataset": dataset_name,
        "count": len(y),
        "mean": y.mean(),
        "std": y.std(),
        "min": y.min(),
        "q25": y.quantile(0.25),
        "median": y.median(),
        "q75": y.quantile(0.75),
        "max": y.max(),
    })

In [ ]:
target_summary_df = pd.DataFrame(target_summary_rows)

print(f"\n=== Target Variable Summary ({TARGET_COL}) ===")

print(target_summary_df.round(6).to_string(index=False))

def duplicate_key_summary(df, name):
    dup_mask = df.duplicated(subset=["WAFER_ID", "STAGE"], keep=False)
    dup_rows = df.loc[dup_mask].copy()
    dup_key_count = dup_rows[["WAFER_ID", "STAGE"]].drop_duplicates().shape[0]

    print(f"\n=== Duplicate Key Check: {name} ===")
    print("total rows           :", len(df))
    print("duplicate row count  :", int(dup_mask.sum()))
    print("duplicate key count  :", int(dup_key_count))

    if len(dup_rows) > 0:
        print("\nTop duplicated keys:")
        print(
            dup_rows[["WAFER_ID", "STAGE"]]
            .value_counts()
            .reset_index(name="count")
            .head(10)
            .to_string(index=False)
        )

duplicate_key_summary(train_compressed, "train_compressed")

duplicate_key_summary(valid_compressed, "valid_compressed")

duplicate_key_summary(test_compressed, "test_compressed")

X_tmp = final_train_dataset.drop(columns=[TARGET_COL], errors="ignore").copy()

all_nan_cols = X_tmp.columns[X_tmp.isna().all()].tolist()

constant_cols = [c for c in X_tmp.columns if X_tmp[c].nunique(dropna=True) <= 1]

numeric_cols = X_tmp.select_dtypes(include=[np.number]).columns.tolist()

near_zero_var_rows = []

In [ ]:
for c in numeric_cols:
    s = pd.to_numeric(X_tmp[c], errors="coerce").dropna()
    if len(s) == 0:
        continue
    if s.std(ddof=0) < 1e-8:
        near_zero_var_rows.append({
            "feature": c,
            "std": float(s.std(ddof=0)),
            "nunique": int(s.nunique(dropna=True))
        })

near_zero_var_df = pd.DataFrame(near_zero_var_rows)

print("\n=== Redundant / Low-Variance Feature Check ===")

print("all-NaN column count      :", len(all_nan_cols))

print("constant column count     :", len(constant_cols))

print("near-zero-variance count  :", len(near_zero_var_df))

if len(all_nan_cols) > 0:
    print("\nAll-NaN columns:")
    print(all_nan_cols[:30])

if len(constant_cols) > 0:
    print("\nConstant columns:")
    print(constant_cols[:30])

if len(near_zero_var_df) > 0:
    print("\nNear-zero-variance columns:")
    print(near_zero_var_df.head(20).to_string(index=False))

X_tmp = final_train_dataset.drop(columns=[TARGET_COL], errors="ignore").copy()

missing_summary_df = pd.DataFrame({
    "feature": X_tmp.columns,
    "missing_count": X_tmp.isna().sum().values,
    "missing_ratio": X_tmp.isna().mean().values,
}).sort_values(["missing_ratio", "missing_count"], ascending=False).reset_index(drop=True)

print("\n=== Missing Value Summary ===")

print(missing_summary_df.head(20).to_string(index=False))

print("\nTotal missing values:", int(X_tmp.isna().sum().sum()))

print("Features with missing values:", int((missing_summary_df["missing_count"] > 0).sum()))

In [ ]:
extreme_target_report = pd.concat([
    train_drop_rows.assign(DATASET="Train", OUTLIER_TYPE="Outside [0,200]"),
    valid_drop_rows.assign(DATASET="Validation", OUTLIER_TYPE="Outside [0,200]"),
    test_drop_rows.assign(DATASET="Test", OUTLIER_TYPE="Outside [0,200]"),
    train_target_outliers[["WAFER_ID", "STAGE", TARGET_COL]].assign(DATASET="Train", OUTLIER_TYPE="Extreme by IQR"),
    valid_target_outliers[["WAFER_ID", "STAGE", TARGET_COL]].assign(DATASET="Validation", OUTLIER_TYPE="Extreme by train bounds"),
], ignore_index=True)

extreme_target_report = extreme_target_report.sort_values(
    ["OUTLIER_TYPE", TARGET_COL], ascending=[True, False]
).reset_index(drop=True)

display(extreme_target_report)

## 7. Exploratory Data Analysis

EDA uses the cleaned, merged training table before feature-level preprocessing and scaling.

Process and consumable features are identified by column-name keywords and ranked by absolute Pearson correlation with the target.

The analysis includes stage-wise target distributions, correlation tables, a heatmap of the top three features from each category, and scatter plots for the top-ranked features.

These associations support exploratory hypotheses; they do not establish causal effects or confirm equipment degradation.

The hypothesis table is sorted by signed correlation. A feature may match both keyword categories.

In [ ]:
eda_results = prepare_eda_tables(train_merged_clean, target_col=TARGET_COL)

eda_df = eda_results["eda_df"]

eda_corr_df = eda_results["eda_corr_df"]

hypothesis_df = eda_results["hypothesis_df"]

top_process = eda_results["top_process"]

top_consumable = eda_results["top_consumable"]

stage_distribution_fig = plot_stage_distribution(eda_df, target_col=TARGET_COL)

stage_distribution_fig.savefig(
    FIGURES_DIR / "stage_target_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

display(eda_corr_df.head(20))

display(hypothesis_df)

eda_heatmap_fig = plot_eda_correlation_heatmap(
    eda_df, top_process, top_consumable, target_col=TARGET_COL,
)

eda_heatmap_fig.savefig(
    FIGURES_DIR / "eda_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

eda_scatter_fig = plot_eda_feature_scatter(
    eda_df, top_process, top_consumable, target_col=TARGET_COL,
)

eda_scatter_fig.savefig(
    FIGURES_DIR / "eda_feature_scatter.png",
    dpi=300,
    bbox_inches="tight",
)


## 8. Modeling Data and Experiment Settings

The target and time-order column are separated from the model features. `START_TIME_ORD` is retained for chronological ordering
and excluded from the predictors.

The experiment uses a random seed of 42, five outer folds, four inner folds, and 15 Optuna trials per model-tuning study.
Target log transformation is disabled.

The time-order column is extracted from the already-cleaned/scaled table rather than raw timestamps. SVR also applies StandardScaler inside its model pipeline. Scatter plots are generated in the plotting section following the original workflow.

In [ ]:
warnings.filterwarnings("ignore")

optuna.logging.set_verbosity(optuna.logging.WARNING)

required_names = [
    "final_train_dataset", "final_valid_dataset", "final_test_dataset",
    "train_label_clean", "valid_label_clean", "test_label_clean",
    "train_keys", "valid_keys", "test_keys"
]

missing_names = [name for name in required_names if name not in globals()]

if missing_names:
    raise ValueError(f"Missing required objects: {missing_names}")

if "TARGET_COL" not in globals():
    TARGET_COL = "AVG_REMOVAL_RATE"

train_dataset = final_train_dataset.copy().reset_index(drop=True)

valid_dataset = final_valid_dataset.copy().reset_index(drop=True)

test_dataset = final_test_dataset.copy().reset_index(drop=True)

if "START_TIME_ORD" not in train_dataset.columns:
    raise ValueError("Missing START_TIME_ORD. Time-series nested CV cannot be performed.")

train_time_order = train_dataset["START_TIME_ORD"].copy().reset_index(drop=True)

valid_time_order = valid_dataset["START_TIME_ORD"].copy().reset_index(drop=True)

test_time_order = test_dataset["START_TIME_ORD"].copy().reset_index(drop=True)

X_train = train_dataset.drop(columns=[TARGET_COL, "START_TIME_ORD"], errors="ignore").select_dtypes(include=[np.number]).copy()

y_train = train_dataset[TARGET_COL].astype(float).reset_index(drop=True)

X_valid = valid_dataset.drop(columns=[TARGET_COL, "START_TIME_ORD"], errors="ignore").select_dtypes(include=[np.number]).copy()

y_valid = valid_dataset[TARGET_COL].astype(float).reset_index(drop=True)

X_test = test_dataset.drop(columns=[TARGET_COL, "START_TIME_ORD"], errors="ignore").select_dtypes(include=[np.number]).copy()

y_test = test_dataset[TARGET_COL].astype(float).reset_index(drop=True)

for c in X_train.columns:
    if c not in X_valid.columns:
        X_valid[c] = np.nan
    if c not in X_test.columns:
        X_test[c] = np.nan

In [ ]:
extra_valid_cols = [c for c in X_valid.columns if c not in X_train.columns]

if len(extra_valid_cols) > 0:
    X_valid = X_valid.drop(columns=extra_valid_cols)

extra_test_cols = [c for c in X_test.columns if c not in X_train.columns]

if len(extra_test_cols) > 0:
    X_test = X_test.drop(columns=extra_test_cols)

X_valid = X_valid[X_train.columns].copy()

X_test = X_test[X_train.columns].copy()

X_train_raw = X_train.copy()

X_valid_raw = X_valid.copy()

X_test_raw = X_test.copy()

train_keys = train_keys.copy().reset_index(drop=True)

valid_keys = valid_keys.copy().reset_index(drop=True)

test_keys = test_keys.copy().reset_index(drop=True)

print("=== Data Shape Check ===")

print("X_train:", X_train.shape)

print("y_train:", y_train.shape)

print("X_valid:", X_valid.shape)

print("y_valid:", y_valid.shape)

print("X_test :", X_test.shape)

print("y_test :", y_test.shape)

RANDOM_STATE = 42

OUTER_FOLDS = 5

INNER_FOLDS = 4

USE_LOG_TARGET = False

N_TRIALS = 15

In [ ]:
N_JOBS = -1

GROUP_CUTOFF = 120.0

ENSEMBLE_WEIGHT_TRIALS = 30

SHOW_FOLD_LOG = False

SHOW_LARGE_TABLES = False

SHOW_SCATTER_PLOTS = True

group_order = ["low", "high"]

base_model_order = ["XGBoost", "RandomForest", "SVR"]

model_order = ["XGBoost", "RandomForest", "SVR", "EnsembleMean", "EnsembleWeighted"]

print("\n=== Config ===")

print("OUTER_FOLDS =", OUTER_FOLDS)

print("INNER_FOLDS =", INNER_FOLDS)

print("USE_LOG_TARGET =", USE_LOG_TARGET)

print("N_TRIALS =", N_TRIALS)

print("GROUP_CUTOFF =", GROUP_CUTOFF)

def prepare_label_group_df(label_df, target_col=TARGET_COL):
    out = label_df.copy()
    out["STAGE"] = out["STAGE"].astype(str).str.strip().str.upper()
    out[target_col] = pd.to_numeric(out[target_col], errors="coerce")
    out = out.dropna(subset=["WAFER_ID", "STAGE", target_col]).reset_index(drop=True)
    return out[["WAFER_ID", "STAGE", target_col]].copy()

### 8.1 Target-Based Grouping

Samples are aligned to label keys and rechecked against [0, 200]. Observed targets below 120 define the low group; values from 120 to 200 define the high group. This rule is applied to training, validation, and test.

Grouping is not chamber-based. Test labels determine which group model predicts each row, so these scores assume known true groups and do not represent end-to-end prediction for samples with unknown targets.

In [ ]:
train_label_group_df = prepare_label_group_df(train_label_clean, target_col=TARGET_COL)

valid_label_group_df = prepare_label_group_df(valid_label_clean, target_col=TARGET_COL)

test_label_group_df = prepare_label_group_df(test_label_clean, target_col=TARGET_COL)

train_label_group_df = train_keys.merge(
    train_label_group_df,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

valid_label_group_df = valid_keys.merge(
    valid_label_group_df,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

test_label_group_df = test_keys.merge(
    test_label_group_df,
    on=["WAFER_ID", "STAGE"],
    how="left"
)

if train_label_group_df[TARGET_COL].isna().any():
    raise ValueError("train_label_group_df has missing target after key alignment.")

if valid_label_group_df[TARGET_COL].isna().any():
    raise ValueError("valid_label_group_df has missing target after key alignment.")

if test_label_group_df[TARGET_COL].isna().any():
    raise ValueError("test_label_group_df has missing target after key alignment.")

train_keep_mask = ((train_label_group_df[TARGET_COL] >= 0) & (train_label_group_df[TARGET_COL] <= 200)).to_numpy()

valid_keep_mask = ((valid_label_group_df[TARGET_COL] >= 0) & (valid_label_group_df[TARGET_COL] <= 200)).to_numpy()

test_keep_mask = ((test_label_group_df[TARGET_COL] >= 0) & (test_label_group_df[TARGET_COL] <= 200)).to_numpy()

if len(train_keep_mask) != len(X_train):
    raise ValueError(f"Train mask length mismatch: mask={len(train_keep_mask)}, X_train={len(X_train)}")

if len(valid_keep_mask) != len(X_valid):
    raise ValueError(f"Valid mask length mismatch: mask={len(valid_keep_mask)}, X_valid={len(X_valid)}")

if len(test_keep_mask) != len(X_test):
    raise ValueError(f"Test mask length mismatch: mask={len(test_keep_mask)}, X_test={len(X_test)}")

print("\n=== Drop MRR > 200 Summary ===")

In [ ]:
print("Train dropped:", int((~train_keep_mask).sum()))

print("Valid dropped:", int((~valid_keep_mask).sum()))

print("Test dropped :", int((~test_keep_mask).sum()))

train_label_group_df = train_label_group_df.iloc[train_keep_mask].reset_index(drop=True)

valid_label_group_df = valid_label_group_df.iloc[valid_keep_mask].reset_index(drop=True)

test_label_group_df = test_label_group_df.iloc[test_keep_mask].reset_index(drop=True)

X_train = X_train.iloc[train_keep_mask].reset_index(drop=True)

y_train = y_train.iloc[train_keep_mask].reset_index(drop=True)

train_time_order = train_time_order.iloc[train_keep_mask].reset_index(drop=True)

train_keys = train_keys.iloc[train_keep_mask].reset_index(drop=True)

X_valid = X_valid.iloc[valid_keep_mask].reset_index(drop=True)

y_valid = y_valid.iloc[valid_keep_mask].reset_index(drop=True)

valid_time_order = valid_time_order.iloc[valid_keep_mask].reset_index(drop=True)

valid_keys = valid_keys.iloc[valid_keep_mask].reset_index(drop=True)

X_test = X_test.iloc[test_keep_mask].reset_index(drop=True)

y_test = y_test.iloc[test_keep_mask].reset_index(drop=True)

test_time_order = test_time_order.iloc[test_keep_mask].reset_index(drop=True)

test_keys = test_keys.iloc[test_keep_mask].reset_index(drop=True)

X_train_raw = X_train.copy()

X_valid_raw = X_valid.copy()

X_test_raw = X_test.copy()

train_label_group_df["group"] = np.where(
    (train_label_group_df[TARGET_COL] >= GROUP_CUTOFF) & (train_label_group_df[TARGET_COL] <= 200),
    "high",
    "low"
)

In [ ]:
valid_label_group_df["group"] = np.where(
    (valid_label_group_df[TARGET_COL] >= GROUP_CUTOFF) & (valid_label_group_df[TARGET_COL] <= 200),
    "high",
    "low"
)

test_label_group_df["group"] = np.where(
    (test_label_group_df[TARGET_COL] >= GROUP_CUTOFF) & (test_label_group_df[TARGET_COL] <= 200),
    "high",
    "low"
)

train_group_map = train_label_group_df[["WAFER_ID", "STAGE", TARGET_COL, "group"]].copy()

valid_group_map = valid_label_group_df[["WAFER_ID", "STAGE", TARGET_COL, "group"]].copy()

test_group_map = test_label_group_df[["WAFER_ID", "STAGE", TARGET_COL, "group"]].copy()

final_train_group = train_group_map["group"].reset_index(drop=True)

valid_group_true = valid_group_map["group"].reset_index(drop=True)

test_group_true = test_group_map["group"].reset_index(drop=True)

if final_train_group.isna().any():
    raise ValueError("final_train_group has NaN after mapping.")

if valid_group_true.isna().any():
    raise ValueError("valid_group_true has NaN after mapping.")

if test_group_true.isna().any():
    raise ValueError("test_group_true has NaN after mapping.")

print("\n=== Shape After Dropping MRR > 200 ===")

print("X_train:", X_train.shape, "y_train:", y_train.shape)

print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)

print("X_test :", X_test.shape, "y_test :", y_test.shape)

## 9. Correlation-Based Feature Prefilter

Pairs with absolute Pearson correlation of at least 0.8 are considered highly correlated.

The retention rule prioritizes fewer missing values, followed by the implemented feature-complexity score and original column order.

This prefilter is fitted on the full training feature matrix before nested cross-validation, matching the original workflow.
Validation and test data use the retained training columns.

Group labels have already been constructed at this point. Existing output text saying Before Grouping refers to the global scope of the filter. Substring conditions in the complexity score are evaluated in order; a name containing _seg1_mean receives the same score as _mean.

In [ ]:
global_prefilter_cols, global_corr_prefilter_df = prefilter_features_by_correlation_global(
    X_df=X_train,
    corr_threshold=0.8,
)

print("\n=== Global Correlation Prefilter Before Grouping ===")

print("Original feature count :", X_train.shape[1])

print("Kept feature count     :", len(global_prefilter_cols))

print("Dropped feature count  :", X_train.shape[1] - len(global_prefilter_cols))

X_train = X_train[global_prefilter_cols].reset_index(drop=True)

X_valid = X_valid[global_prefilter_cols].reset_index(drop=True)

X_test = X_test[global_prefilter_cols].reset_index(drop=True)

X_train_raw = X_train.copy()

X_valid_raw = X_valid.copy()

X_test_raw = X_test.copy()

if len(global_corr_prefilter_df) > 0:
    global_corr_prefilter_df["stage"] = "global_prefilter_before_grouping"
    global_corr_prefilter_df.to_csv(TABLES_DIR / "global_corr_prefilter_decisions.csv", index=False)

print("\n=== Group Summary By Fixed Cutoff ===")

train_group_summary = (
    train_label_group_df
    .groupby("group", as_index=False)
    .agg(
        count=(TARGET_COL, "size"),
        min=(TARGET_COL, "min"),
        median=(TARGET_COL, "median"),
        max=(TARGET_COL, "max"),
        mean=(TARGET_COL, "mean"),
        std=(TARGET_COL, "std"),
    )
)

train_group_summary["ratio"] = train_group_summary["count"] / len(train_label_group_df)

print(train_group_summary.round(6).to_string(index=False))

train_group_summary.to_csv(TABLES_DIR / "train_group_summary_fixed_cutoff.csv", index=False)

### 9.1 Search-Space Reference

The following table describes the ranges implemented in model_building.suggest_params. Editing this display table alone does not change the tuning search space.

In [ ]:
search_space_df = pd.DataFrame([
    {"model": "XGBoost", "group": "low", "param": "n_estimators", "setting": "int, 650 ~ 900"},
    {"model": "XGBoost", "group": "low", "param": "max_depth", "setting": "int, 2 ~ 4"},
    {"model": "XGBoost", "group": "low", "param": "learning_rate", "setting": "float(log), 0.015 ~ 0.04"},
    {"model": "XGBoost", "group": "low", "param": "colsample_bytree", "setting": "float, 0.80 ~ 1.00"},

    {"model": "XGBoost", "group": "high", "param": "n_estimators", "setting": "int, 500 ~ 750"},
    {"model": "XGBoost", "group": "high", "param": "max_depth", "setting": "int, 2 ~ 3"},
    {"model": "XGBoost", "group": "high", "param": "learning_rate", "setting": "float(log), 0.015 ~ 0.04"},
    {"model": "XGBoost", "group": "high", "param": "colsample_bytree", "setting": "float, 0.85 ~ 1.00"},


    {"model": "RandomForest", "group": "low", "param": "n_estimators", "setting": "int, 250 ~ 380"},
    {"model": "RandomForest", "group": "low", "param": "max_depth", "setting": "int, 12 ~ 22"},
    {"model": "RandomForest", "group": "low", "param": "min_samples_split", "setting": "int, 2 ~ 6"},
    {"model": "RandomForest", "group": "low", "param": "max_features", "setting": "float, 0.25 ~ 0.45"},

    {"model": "RandomForest", "group": "high", "param": "n_estimators", "setting": "int, 220 ~ 340"},
    {"model": "RandomForest", "group": "high", "param": "max_depth", "setting": "int, 6 ~ 12"},
    {"model": "RandomForest", "group": "high", "param": "min_samples_split", "setting": "int, 2 ~ 4"},
    {"model": "RandomForest", "group": "high", "param": "max_features", "setting": "float, 0.20 ~ 0.40"},

    {"model": "SVR", "group": "low", "param": "C", "setting": "float(log), 30 ~ 100"},
    {"model": "SVR", "group": "low", "param": "epsilon", "setting": "float(log), 0.003 ~ 0.02"},
    {"model": "SVR", "group": "low", "param": "gamma", "setting": "float(log), 0.001 ~ 0.01"},

    {"model": "SVR", "group": "high", "param": "C", "setting": "float(log), 3 ~ 20"},
    {"model": "SVR", "group": "high", "param": "epsilon", "setting": "float(log), 0.02 ~ 0.08"},
    {"model": "SVR", "group": "high", "param": "gamma", "setting": "float(log), 2e-4 ~ 2e-3"},
])

print("\n=== Hyperparameter Search Space ===")

print(search_space_df.to_string(index=False))

X_train = X_train_raw.copy()

X_valid = X_valid_raw.copy()

X_test = X_test_raw.copy()

## 10. Nested Cross-Validation and Final Training

Separate XGBoost, Random Forest, and SVR models are trained for the low and high groups.

Rows are sorted within each group using the supplied, preprocessed `START_TIME_ORD` values. Both outer and inner validation use fixed-length sliding windows; this ordering may differ from raw chronology if time values were changed during cleaning.

XGBoost-based feature selection is fitted within the relevant training folds. Features are retained until cumulative importance
reaches 80%, with a minimum of eight features when available.

Optuna minimizes mean inner-fold RMSE plus a penalty for overly flat predictions. The result field `inner_best_rmse` contains
this penalized objective rather than plain RMSE.

After outer evaluation, feature selection, tuning, and model fitting are repeated using the full training data of each group.

The penalty is 0.8 * max(0, 0.5 * std(actual) - std(predicted)). Both inner_best_rmse and best_inner_rmse_train contain this penalized objective. Integer rounding means the realized window ratio can differ from 0.8/0.2. Earlier whole-training preprocessing is not refitted inside each fold.

In [ ]:
training_results = train_groupwise_models(
    X_train=X_train, y_train=y_train,
    final_train_group=final_train_group, train_time_order=train_time_order,
    group_order=group_order, base_model_order=base_model_order,
    outer_folds=OUTER_FOLDS, inner_folds=INNER_FOLDS,
    n_trials=N_TRIALS, random_state=RANDOM_STATE,
    use_log_target=USE_LOG_TARGET, n_jobs=N_JOBS,
    show_fold_log=SHOW_FOLD_LOG,
)

In [ ]:
group_model_artifacts = training_results["group_model_artifacts"]

group_selected_features = training_results["group_selected_features"]

group_outer_result_df = training_results["group_outer_result_df"]

group_param_df = training_results["group_param_df"]

feature_selection_df = training_results["feature_selection_df"]

feature_correlation_df = build_feature_correlation_table(
    X_df=X_train,
    y_series=y_train,
    group_labels=final_train_group, group_order=group_order
)

feature_correlation_df["selected_in_low"] = feature_correlation_df["feature"].isin(group_selected_features["low"])

feature_correlation_df["selected_in_high"] = feature_correlation_df["feature"].isin(group_selected_features["high"])

feature_selection_df.to_csv(TABLES_DIR / "group_feature_selection_importance_80pct.csv", index=False)

feature_correlation_df.to_csv(TABLES_DIR / "feature_target_correlation_table.csv", index=False)

if SHOW_LARGE_TABLES:
    print(feature_correlation_df.head(20).round(6).to_string(index=False))

    print("\n=== Group Nested CV Results ===")
    print(group_outer_result_df)

print("\n=== Group Final Refit Params ===")

print(group_param_df)

## 11. Group-Routed Predictions and Ensembles

Each sample is routed to the model corresponding to its supplied target-based group.

EnsembleMean averages the predictions of the three base models.
EnsembleWeighted uses group-specific non-negative weights normalized to sum to one.

The weights are jointly optimized using overall validation RMSE with 30 Optuna trials and seed 2068. The learned weights are then applied to training, validation, and test predictions.

The validation_rmse value repeated in each group row of the weight table is the overall optimization objective, not a separate group RMSE.

In [ ]:
train_pred_map = build_prediction_map(
    X_train, final_train_group, group_model_artifacts, group_selected_features,
    base_model_order=base_model_order, group_order=group_order,
    use_log_target=USE_LOG_TARGET,
)

valid_pred_map = build_prediction_map(
    X_valid, valid_group_true, group_model_artifacts, group_selected_features,
    base_model_order=base_model_order, group_order=group_order,
    use_log_target=USE_LOG_TARGET,
)

test_pred_map = build_prediction_map(
    X_test, test_group_true, group_model_artifacts, group_selected_features,
    base_model_order=base_model_order, group_order=group_order,
    use_log_target=USE_LOG_TARGET,
)

ensemble_weights_by_group, ensemble_weight_df = tune_groupwise_ensemble_weights(
    valid_pred_map=valid_pred_map,
    y_valid_true=y_valid.values,
    valid_group_labels=valid_group_true.values,
    model_names=base_model_order,
    n_trials=ENSEMBLE_WEIGHT_TRIALS,
    seed=RANDOM_STATE + 2026,
    group_order=group_order,
)

train_pred_map["EnsembleWeighted"] = apply_weighted_ensemble(
    pred_map=train_pred_map,
    group_labels=final_train_group.values,
    weights_by_group=ensemble_weights_by_group,
    model_names=base_model_order,
    group_order=group_order,
)

valid_pred_map["EnsembleWeighted"] = apply_weighted_ensemble(
    pred_map=valid_pred_map,
    group_labels=valid_group_true.values,
    weights_by_group=ensemble_weights_by_group,
    model_names=base_model_order,
    group_order=group_order,
)

In [ ]:
test_pred_map["EnsembleWeighted"] = apply_weighted_ensemble(
    pred_map=test_pred_map,
    group_labels=test_group_true.values,
    weights_by_group=ensemble_weights_by_group,
    model_names=base_model_order,
    group_order=group_order,
)

print("\n=== Ensemble Weights Learned On Validation ===")

print(ensemble_weight_df.round(6).to_string(index=False))

## 12. Model Evaluation

Models are compared using RMSE, MAE, and MSE on the original removal-rate scale.

The reported `mse_std` is the sample standard deviation of individual squared errors. It is not the standard deviation of MSE across cross-validation folds.

Training-set errors describe fitted-model performance.
Validation results also reflect its use in ensemble-weight optimization.

Test scores concern a target-filtered population and assume known target-derived groups. Group A and Group B in formatted MSE tables refer to low/high groups, not STAGE A/B.

In [ ]:
validation_eval_df = pd.DataFrame([
    {
        "model": model_name,
        "rmse": rmse(y_valid.values, valid_pred_map[model_name]),
        "mae": mae(y_valid.values, valid_pred_map[model_name]),
        "mse": mse(y_valid.values, valid_pred_map[model_name]),
        "mse_std": squared_error_stats(y_valid.values, valid_pred_map[model_name])["mse_std"],
    }
    for model_name in model_order
]).sort_values(["rmse", "mae"]).reset_index(drop=True)

test_eval_df = pd.DataFrame([
    {
        "model": model_name,
        "rmse": rmse(y_test.values, test_pred_map[model_name]),
        "mae": mae(y_test.values, test_pred_map[model_name]),
        "mse": mse(y_test.values, test_pred_map[model_name]),
        "mse_std": squared_error_stats(y_test.values, test_pred_map[model_name])["mse_std"],
    }
    for model_name in model_order
]).sort_values(["rmse", "mae"]).reset_index(drop=True)

print("\n=== Validation Evaluation ===")

print(validation_eval_df)

print("\n=== Final Test Evaluation ===")

print(test_eval_df)

overall_mse_table = (
    validation_eval_df[["model", "mse", "mse_std"]]
    .rename(columns={"mse": "Validation_Overall_MSE", "mse_std": "Validation_Overall_MSE_STD"})
    .merge(
        test_eval_df[["model", "mse", "mse_std"]]
        .rename(columns={"mse": "Test_Overall_MSE", "mse_std": "Test_Overall_MSE_STD"}),
        on="model",
        how="inner"
    )
)

overall_mse_table["Validation_Overall_MSE"] = overall_mse_table.apply(
    lambda r: f"{r['Validation_Overall_MSE']:.4f}({r['Validation_Overall_MSE_STD']:.4f})",
    axis=1
)

overall_mse_table["Test_Overall_MSE"] = overall_mse_table.apply(
    lambda r: f"{r['Test_Overall_MSE']:.4f}({r['Test_Overall_MSE_STD']:.4f})",
    axis=1
)

In [ ]:
overall_mse_table = overall_mse_table[[
    "model",
    "Validation_Overall_MSE",
    "Test_Overall_MSE"
]]

print("\n=== Overall MSE Table ===")

print(overall_mse_table.to_string(index=False))

group_mse_rows = []

for g in group_order:
    train_idx = np.where(final_train_group == g)[0]
    valid_idx = np.where(valid_group_true == g)[0]
    test_idx = np.where(test_group_true == g)[0]

    for model_name in model_order:
        train_stats = squared_error_stats(
            y_train.iloc[train_idx].values,
            train_pred_map[model_name][train_idx]
        ) if len(train_idx) > 0 else {"mse_mean": np.nan, "mse_std": np.nan}

        valid_stats = squared_error_stats(
            y_valid.iloc[valid_idx].values,
            valid_pred_map[model_name][valid_idx]
        ) if len(valid_idx) > 0 else {"mse_mean": np.nan, "mse_std": np.nan}

        test_stats = squared_error_stats(
            y_test.iloc[test_idx].values,
            test_pred_map[model_name][test_idx]
        ) if len(test_idx) > 0 else {"mse_mean": np.nan, "mse_std": np.nan}

        group_mse_rows.append({
            "group": g,
            "model": model_name,
            "Training_MSE_Mean": train_stats["mse_mean"],
            "Training_MSE_STD": train_stats["mse_std"],
            "Validation_MSE_Mean": valid_stats["mse_mean"],
            "Validation_MSE_STD": valid_stats["mse_std"],
            "Final_Test_MSE_Mean": test_stats["mse_mean"],
            "Final_Test_MSE_STD": test_stats["mse_std"],
        })

group_mse_df = pd.DataFrame(group_mse_rows)

In [ ]:
for model_name in model_order:
    table_df = (
        group_mse_df[group_mse_df["model"] == model_name][[
            "group",
            "Training_MSE_Mean", "Training_MSE_STD",
            "Validation_MSE_Mean", "Validation_MSE_STD",
            "Final_Test_MSE_Mean", "Final_Test_MSE_STD"
        ]]
        .copy()
        .reset_index(drop=True)
    )

    table_df["Metric"] = table_df["group"].map({
        "low": "GroupA MSE",
        "high": "GroupB MSE",
    })

    table_df["Training_MSE"] = table_df.apply(
        lambda r: f"{r['Training_MSE_Mean']:.4f}({r['Training_MSE_STD']:.4f})", axis=1
    )
    table_df["Validation_MSE"] = table_df.apply(
        lambda r: f"{r['Validation_MSE_Mean']:.4f}({r['Validation_MSE_STD']:.4f})", axis=1
    )
    table_df["Testing_MSE"] = table_df.apply(
        lambda r: f"{r['Final_Test_MSE_Mean']:.4f}({r['Final_Test_MSE_STD']:.4f})", axis=1
    )

    table_df = table_df[["Metric", "Training_MSE", "Validation_MSE", "Testing_MSE"]]

    print(f"\n=== MSE Table ({model_name}) ===")
    print(table_df.to_string(index=False))

## 13. Model Interpretation

Feature importance is calculated separately for each model and group:

- XGBoost: gain importance.
- Random Forest: impurity-based importance.
- SVR: permutation importance using negative MSE and five repeats.

SVR permutation importance is evaluated on the corresponding training group. Importance values have different definitions across models and should not be compared directly in magnitude.

In [ ]:
group_feature_importance = {}

for g in group_order:
    idx = np.where(final_train_group == g)[0]
    selected_cols = group_selected_features[g]
    X_g = X_train.iloc[idx][selected_cols].reset_index(drop=True)
    y_g = y_train.iloc[idx].reset_index(drop=True)
    time_g = train_time_order.iloc[idx].reset_index(drop=True)

    sort_idx = np.argsort(time_g.values)
    X_g = X_g.iloc[sort_idx].reset_index(drop=True)
    y_g = y_g.iloc[sort_idx].reset_index(drop=True)
    y_g_model = transform_target(y_g, use_log_target=USE_LOG_TARGET)

    group_feature_importance[g] = {}

    for model_name in base_model_order:
        fitted_obj = group_model_artifacts[model_name][g]

        if model_name == "XGBoost":
            imp = get_xgb_feature_importance(fitted_obj, X_g.columns)
        elif model_name == "RandomForest":
            imp = get_rf_feature_importance(fitted_obj, X_g.columns)
        else:
            imp = get_svr_permutation_importance(
                fitted_obj,
                X_eval=X_g,
                y_eval_true_model_scale=y_g_model.values,
                random_state=RANDOM_STATE,
            )

        group_feature_importance[g][model_name] = imp

### 13.1 Actual-versus-Predicted Plots

These plots visualize the same target-conditioned evaluation as the metric tables. The supplied groups are low/high target groups. The plotting function uses blue/red by default.

In [ ]:
group_color_map = {
    "low": "blue",
    "high": "red",
}

prediction_scatter_figures = {}

for model_name in model_order:
    fig = plot_scatter(
        y_valid.values,
        valid_group_true,
        valid_pred_map[model_name],
        "Validation Set",
        model_name,
    )
    prediction_scatter_figures[f"validation_{model_name}"] = fig
    fig.savefig(
        FIGURES_DIR / f"validation_{model_name.lower()}_actual_vs_predicted.png",
        dpi=300,
        bbox_inches="tight",
    )

for model_name in model_order:
    fig = plot_scatter(
        y_test.values,
        test_group_true,
        test_pred_map[model_name],
        "Final Test Set",
        model_name,
    )
    prediction_scatter_figures[f"test_{model_name}"] = fig
    fig.savefig(
        FIGURES_DIR / f"test_{model_name.lower()}_actual_vs_predicted.png",
        dpi=300,
        bbox_inches="tight",
    )


## 14. Export Results

Exported tables include cross-validation results, selected features, tuned parameters, group mappings, model comparisons,
ensemble weights, and sample-level predictions.

CSV files are written to `results/tables/`, and generated figures are written to `results/figures/`.

Some intermediate tables are exported earlier. The correlation-decision CSV is only written when pairs are removed, although the printed file list always names it. Prediction CSVs omit wafer-stage keys; use the matching group-mapping CSV in the same row order.


In [ ]:
group_outer_result_df.to_csv(TABLES_DIR / "group_nested_cv_outer_results.csv", index=False)

group_param_df.to_csv(TABLES_DIR / "group_model_params_bayes.csv", index=False)

validation_eval_df.to_csv(TABLES_DIR / "group_validation_model_comparison.csv", index=False)

test_eval_df.to_csv(TABLES_DIR / "group_final_test_model_comparison.csv", index=False)

group_mse_df.to_csv(TABLES_DIR / "groupwise_mse_table_with_std.csv", index=False)

search_space_df.to_csv(TABLES_DIR / "model_search_space.csv", index=False)

train_group_map.to_csv(TABLES_DIR / "train_group_mapping.csv", index=False)

valid_group_map.to_csv(TABLES_DIR / "validation_group_mapping.csv", index=False)

test_group_map.to_csv(TABLES_DIR / "test_group_mapping.csv", index=False)

valid_pred_df = pd.DataFrame({
    "y_valid_true": y_valid.values,
    "group": valid_group_true.values,
    "pred_xgboost": valid_pred_map["XGBoost"],
    "pred_randomforest": valid_pred_map["RandomForest"],
    "pred_svr": valid_pred_map["SVR"],
    "pred_ensemble_mean": valid_pred_map["EnsembleMean"],
    "pred_ensemble_weighted": valid_pred_map["EnsembleWeighted"],
})

valid_pred_df.to_csv(TABLES_DIR / "validation_predictions.csv", index=False)

test_pred_df = pd.DataFrame({
    "y_test_true": y_test.values,
    "group": test_group_true.values,
    "pred_xgboost": test_pred_map["XGBoost"],
    "pred_randomforest": test_pred_map["RandomForest"],
    "pred_svr": test_pred_map["SVR"],
    "pred_ensemble_mean": test_pred_map["EnsembleMean"],
    "pred_ensemble_weighted": test_pred_map["EnsembleWeighted"],
})

test_pred_df.to_csv(TABLES_DIR / "final_test_predictions.csv", index=False)

ensemble_weight_df.to_csv(TABLES_DIR / "ensemble_group_weights.csv", index=False)

print("\n=== Saved Files ===")

print("- train_group_summary_fixed_cutoff.csv")

print("- model_search_space.csv")

In [ ]:
print("- group_feature_selection_importance_80pct.csv")

print("- global_corr_prefilter_decisions.csv")

print("- feature_target_correlation_table.csv")

print("- group_nested_cv_outer_results.csv")

print("- group_model_params_bayes.csv")

print("- group_validation_model_comparison.csv")

print("- group_final_test_model_comparison.csv")

print("- groupwise_mse_table_with_std.csv")

print("- train_group_mapping.csv")

print("- validation_group_mapping.csv")

print("- test_group_mapping.csv")

print("- validation_predictions.csv")

print("- final_test_predictions.csv")

print("- ensemble_group_weights.csv")

print("- feature importance kept in memory: group_feature_importance")

### 14.1 Additional Tables and Feature Summaries

The remaining cells display further model comparisons, top-15 feature importance plots, and selection summaries. They do not retrain models.

In [ ]:
model_mse_tables = {}

for model_name in model_order:
    table_df = (
        group_mse_df[group_mse_df["model"] == model_name][[
            "group",
            "Training_MSE_Mean", "Training_MSE_STD",
            "Validation_MSE_Mean", "Validation_MSE_STD",
            "Final_Test_MSE_Mean", "Final_Test_MSE_STD"
        ]]
        .copy()
        .reset_index(drop=True)
    )

    table_df["Metric"] = table_df["group"].map({
        "low": "GroupA MSE",
        "high": "GroupB MSE",
    })

    table_df = table_df[[
        "Metric",
        "Training_MSE_Mean", "Training_MSE_STD",
        "Validation_MSE_Mean", "Validation_MSE_STD",
        "Final_Test_MSE_Mean", "Final_Test_MSE_STD"
    ]]

    model_mse_tables[model_name] = table_df

    print(f"\n=== MSE Table ({model_name}) ===")
    print(table_df.round(4).to_string(index=False))

overall_mse_df = pd.DataFrame([
    {
        "model": model_name,
        "Validation_Overall_MSE": mse(y_valid.values, valid_pred_map[model_name]),
        "Test_Overall_MSE": mse(y_test.values, test_pred_map[model_name]),
    }
    for model_name in model_order
]).sort_values("Validation_Overall_MSE").reset_index(drop=True)

display(overall_mse_df.round(4))

importance_figures = plot_group_feature_importance(group_feature_importance, base_model_order=base_model_order, group_order=group_order, top_n=15)

for model_name, fig in importance_figures.items():
    fig.savefig(
        FIGURES_DIR / f"feature_importance_{model_name.lower()}.png",
        dpi=300,
        bbox_inches="tight",
    )

print("=== Feature Count Summary ===")

print("After global correlation prefilter:")

print("low group  :", len(global_prefilter_cols))

In [ ]:
print("high group :", len(global_prefilter_cols))

print("\nAfter XGBoost cumulative importance 80%:")

print("low group  :", len(group_selected_features["low"]))

print("high group :", len(group_selected_features["high"]))

low_detail = feature_selection_df[
    (feature_selection_df["group"] == "low") &
    (feature_selection_df["outer_fold"] == "final_refit") &
    (feature_selection_df["selected"] == True)
].copy()

high_detail = feature_selection_df[
    (feature_selection_df["group"] == "high") &
    (feature_selection_df["outer_fold"] == "final_refit") &
    (feature_selection_df["selected"] == True)
].copy()

print("low group final selected cumulative importance ratio:",
      low_detail["cumulative_importance"].max())

print("high group final selected cumulative importance ratio:",
      high_detail["cumulative_importance"].max())

print("\n=== Global Correlation Prefilter Table ===")

display(global_corr_prefilter_df)

## 15. Limitations and Reproducibility

This notebook preserves the original modeling workflow while organizing the code into reusable Python modules.

Key methodological limitations include target-based routing, target-range filtering of evaluation samples, and preprocessing and correlation
prefiltering performed before nested cross-validation.

Feature associations and model importance provide exploratory evidence rather than causal or diagnostic confirmation.

The time-order column is altered by feature preprocessing before sorting. The split does not enforce a time gap or wafer-level separation. Interpolation can use neighboring or later feature rows, and whole-training preprocessing can influence CV evaluation folds.

The workflow assumes that each target group has enough samples for the configured time-series splits and that the feature columns retained during training are available after validation and test alignment.
